# 1. Data Understanding

## 1.1 Campaigns

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
Path.cwd()

WindowsPath('C:/Users/marta/OneDrive/Desktop/PROJEKTY/retail-promotion-effectiveness/notebooks')

In [5]:
campaign_desc = pd.read_csv("../data/raw/campaign_desc.csv")

In [6]:
campaign_desc.head()

,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691
3,TypeC,20,615,685
4,TypeB,23,646,684


In [7]:
campaign_desc.shape

(30, 4)

In [8]:
campaign_desc.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   DESCRIPTION  30 non-null     str  
 1   CAMPAIGN     30 non-null     int64
 2   START_DAY    30 non-null     int64
 3   END_DAY      30 non-null     int64
dtypes: int64(3), str(1)
memory usage: 1.1 KB


In [9]:
campaign_desc["CAMPAIGN"].is_unique

True

In [10]:
campaign_desc["DESCRIPTION"].value_counts()

DESCRIPTION
TypeB    19
TypeC     6
TypeA     5
Name: count, dtype: int64

In [11]:
campaign_desc[["START_DAY", "END_DAY"]].agg(["min", "max"])

,START_DAY,END_DAY
min,224,264
max,659,719


In [12]:
# Calculate campaign duration (inclusive of start and end days)
campaign_desc["DURATION_DAYS"] = campaign_desc["END_DAY"] - campaign_desc["START_DAY"] + 1

In [13]:
# Summarize campaign duration by campaign type
campaign_desc.groupby("DESCRIPTION")["DURATION_DAYS"].agg(["count", "min", "median", "max"])

,count,min,median,max
DESCRIPTION,,,,
TypeA,5,41,48.0,56
TypeB,19,33,33.0,62
TypeC,6,33,65.0,162


In [14]:
# Check for campaigns with zero or negative duration
(campaign_desc["DURATION_DAYS"] <= 0).sum()

np.int64(0)

In [15]:
# Identify campaign pairs with overlapping periods
pairs = campaign_desc.merge(campaign_desc, how="cross", suffixes=("_1", "_2"))

overlaps = pairs[
    (pairs["CAMPAIGN_1"] < pairs["CAMPAIGN_2"])
    & (pairs["START_DAY_1"] <= pairs["END_DAY_2"])
    & (pairs["START_DAY_2"] <= pairs["END_DAY_1"])
]

len(overlaps)

74

In [16]:
# Calculate the number of overlapping days for each campaign pair
overlaps = overlaps.copy()

overlaps["OVERLAP_DAYS"] = (
    overlaps[["END_DAY_1", "END_DAY_2"]].min(axis=1)
    - overlaps[["START_DAY_1", "START_DAY_2"]].max(axis=1)
    + 1
)

In [18]:
# Identify campaign pairs with the longest overlapping periods
overlaps[["CAMPAIGN_1", "CAMPAIGN_2", "OVERLAP_DAYS"]].sort_values("OVERLAP_DAYS", ascending=False).head(10)

,CAMPAIGN_1,CAMPAIGN_2,OVERLAP_DAYS
33,15,20,71
37,15,18,56
30,15,24,50
301,14,15,50
867,27,28,42
836,28,29,40
94,20,23,39
34,15,23,39
621,3,5,35
2,24,25,33


Key findings:

- 30 unique campaigns: 5 TypeA, 19 TypeB, and 6 TypeC.

- Median duration: 48 days (TypeA), 33 days (TypeB), and 65 days (TypeC).

- No missing values or invalid campaign durations.

- 74 campaign pairs overlap in time, with a maximum overlap of 71 days.

- Temporal overlap does not necessarily imply overlapping household participation.

## 1.2 Campaign Participation

In [20]:
campaign_table = pd.read_csv("../data/raw/campaign_table.csv")

In [23]:
campaign_table.head()

,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26
3,TypeA,208,26
4,TypeA,192,26


In [25]:
campaign_table.shape

(7208, 3)

In [26]:
campaign_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 7208 entries, 0 to 7207
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   DESCRIPTION    7208 non-null   str  
 1   household_key  7208 non-null   int64
 2   CAMPAIGN       7208 non-null   int64
dtypes: int64(2), str(1)
memory usage: 169.1 KB


In [27]:
# Check for duplicate household-campaign assignments
campaign_table.duplicated(subset=["household_key", "CAMPAIGN"]).sum()

np.int64(0)

In [28]:
campaign_table["household_key"].nunique()

1584

In [29]:
# Summarize the number of campaigns assigned to each household
campaign_table.groupby("household_key")["CAMPAIGN"].nunique().describe()

count    1584.000000
mean        4.550505
std         2.993988
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max        17.000000
Name: CAMPAIGN, dtype: float64

In [30]:
# Count unique households assigned to each campaign
campaign_sizes = campaign_table.groupby("CAMPAIGN")["household_key"].nunique()

campaign_sizes.describe()

count      30.000000
mean      240.266667
std       304.411579
min        12.000000
25%        69.000000
50%       173.000000
75%       221.500000
max      1133.000000
Name: household_key, dtype: float64

In [31]:
# Identify campaigns with the largest number of households
campaign_sizes.sort_values(ascending=False).head(10)

CAMPAIGN
18    1133
13    1077
8     1076
30     361
26     332
22     276
20     244
14     224
11     214
17     202
Name: household_key, dtype: int64

In [32]:
# Check for campaign IDs missing from the campaign description table
(~campaign_table["CAMPAIGN"].isin(campaign_desc["CAMPAIGN"])).sum()

np.int64(0)

In [33]:
# Create unique pairs of campaigns assigned to the same household
household_pairs = campaign_table.merge(campaign_table, on="household_key",suffixes=("_1", "_2"))
household_pairs = household_pairs[household_pairs["CAMPAIGN_1"] < household_pairs["CAMPAIGN_2"]]

In [34]:
# Identify households assigned to campaigns with overlapping periods
household_overlaps = household_pairs.merge(overlaps[["CAMPAIGN_1", "CAMPAIGN_2", "OVERLAP_DAYS"]], on=["CAMPAIGN_1", "CAMPAIGN_2"], how="inner")

In [35]:
# Count households assigned to overlapping campaigns
household_overlaps["household_key"].nunique()

884

In [36]:
# Count overlapping campaign pairs per household
overlap_counts = household_overlaps.groupby("household_key").size()
overlap_counts.describe()

count    884.000000
mean       3.665158
std        3.307031
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max       20.000000
dtype: float64

In [37]:
# Identify households with the most overlapping campaign pairs
overlap_counts.sort_values(ascending=False).head(10)

household_key
2489    20
982     19
2459    19
371     18
2317    17
1453    16
1803    16
676     16
1527    15
2341    15
dtype: int64

**Key findings:**
- 7,208 campaign assignments across 1,584 unique households, with no duplicate household-campaign pairs.
- Households participated in a median of 4 campaigns (maximum: 17).
- Campaign size ranged from 12 to 1,133 households.
- 884 households (55.8%) were assigned to campaigns with overlapping periods.
- Among these households, the median number of overlapping campaign pairs was 2 (maximum: 20)
- Campaign overlap: 884 households (55.8%) were assigned to campaigns with overlapping validity periods. Actual offer receipt dates are unavailable.

## 1.3 Coupons

In [39]:
coupon = pd.read_csv("../data/raw/coupon.csv")

In [40]:
coupon.head()

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
0,10000089061,27160,4
1,10000089064,27754,9
2,10000089073,28897,12
3,51800009050,28919,28
4,52100000076,28929,25


In [41]:
coupon.shape

(124548, 3)

In [42]:
coupon.info()

<class 'pandas.DataFrame'>
RangeIndex: 124548 entries, 0 to 124547
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   COUPON_UPC  124548 non-null  int64
 1   PRODUCT_ID  124548 non-null  int64
 2   CAMPAIGN    124548 non-null  int64
dtypes: int64(3)
memory usage: 2.9 MB


In [43]:
# Check for duplicate campaign-coupon-product combinations
coupon.duplicated(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).sum()

np.int64(5164)

In [44]:
# Count unique campaign-coupon combinations
coupon[["CAMPAIGN", "COUPON_UPC"]].drop_duplicates().shape[0]

1397

In [45]:
# Summarize the number of eligible products per campaign-coupon
products_per_coupon = coupon.groupby(["CAMPAIGN", "COUPON_UPC"])["PRODUCT_ID"].nunique()
products_per_coupon.describe()

count     1397.000000
mean        85.457409
std        688.682458
min          1.000000
25%          6.000000
50%         13.000000
75%         30.000000
max      14477.000000
Name: PRODUCT_ID, dtype: float64

In [46]:
# Inspect duplicate campaign-coupon-product records
coupon[coupon.duplicated(keep=False)].sort_values(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).head(10)

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
52212,50004110076,133534,8
52213,50004110076,133534,8
49849,50004110076,140705,8
49850,50004110076,140705,8
31573,50004110076,849713,8
31574,50004110076,849713,8
52210,50004110076,862863,8
52211,50004110076,862863,8
53123,50004110076,870248,8
53124,50004110076,870248,8


In [47]:
# Count duplicate records by campaign
coupon.loc[coupon.duplicated(), "CAMPAIGN"].value_counts()

CAMPAIGN
27    3391
8      608
30     576
26     292
13     173
18     109
14      13
29       2
Name: count, dtype: int64

In [48]:
# Identify the campaign-coupon combination with the most eligible products
products_per_coupon.sort_values(ascending=False).head(10)

CAMPAIGN  COUPON_UPC 
18        10000085478    14477
13        10000085428    14367
          10000085427     5713
18        10000085477     5686
          10000085479     5474
13        10000085429     5423
8         10000085363     4538
18        10000085475     4294
8         10000085361     4241
13        10000085425     4189
Name: PRODUCT_ID, dtype: int64

In [49]:
# Check the frequency of repeated campaign-coupon-product combinations
coupon.groupby(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).size().value_counts().sort_index()

1     117560
2        688
3        304
4        384
5        173
6         97
7         74
8         18
9         10
10         8
11         7
12        38
13         1
14         7
15         7
16         4
17         2
19         2
Name: count, dtype: int64

In [50]:
# Inspect the highest duplicate frequencies
coupon.groupby(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).size().value_counts().sort_index().tail(8)

11     7
12    38
13     1
14     7
15     7
16     4
17     2
19     2
Name: count, dtype: int64

In [51]:
# Identify the most frequently repeated campaign-coupon-product combinations
coupon.groupby(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).size().sort_values(ascending=False).head(10)

CAMPAIGN  COUPON_UPC   PRODUCT_ID
27        57940011080  921912        19
          57940011075  921912        19
          57940018081  6704447       17
          57940018076  6704447       17
          57940018081  7167488       16
          54589314187  1024433       16
          54589314175  1024433       16
          57940018076  7167488       16
          54100027032  1118641       15
          51111122275  5565415       15
dtype: int64

In [52]:
# Check for coupon records referencing unknown campaigns
(~coupon["CAMPAIGN"].isin(campaign_desc["CAMPAIGN"])).sum()

np.int64(0)

In [53]:
# Count distinct coupons per campaign
coupons_per_campaign = coupon.groupby("CAMPAIGN")["COUPON_UPC"].nunique()
coupons_per_campaign.describe()

count     30.000000
mean      46.566667
std       69.301781
min        1.000000
25%       12.250000
50%       17.500000
75%       27.750000
max      209.000000
Name: COUPON_UPC, dtype: float64

In [54]:
# Identify campaigns with the most distinct coupons
coupons_per_campaign.sort_values(ascending=False).head(10)

CAMPAIGN
8     209
18    209
13    207
26    181
30    181
3      34
29     33
28     28
27     27
14     26
Name: COUPON_UPC, dtype: int64

**Key findings:**
- 124,548 records representing 1,397 unique campaign-coupon combinations.
- Campaigns contained 1–209 distinct coupons (median: 17.5).
- Each coupon covered a median of 13 products, with a maximum of 14,477.
- 5,164 excess duplicate records were identified across 8 campaigns, with individual combinations repeated up to 19 times.
- All campaign IDs match `campaign_desc`.
- Duplicate coupon-product mappings require deduplication before analytical joins.